In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import torch
import torch.nn as nn
import numpy as np
from torchvision import transforms
from transformers import TimesformerModel, TimesformerConfig


In [ ]:
class FightTimeSformer(nn.Module):
    def __init__(self, backbone, num_classes=2):
        super().__init__()
        self.backbone = backbone
        self.classifier = nn.Linear(
            backbone.config.hidden_size,
            num_classes
        )

    def forward(self, x):
        outputs = self.backbone(x)
        cls_token = outputs.last_hidden_state[:, 0]
        return self.classifier(cls_token)


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

config = TimesformerConfig.from_pretrained(
    "facebook/timesformer-base-finetuned-k400"
)
config.num_frames = 8
config.image_size = 224

backbone = TimesformerModel.from_pretrained(
    "facebook/timesformer-base-finetuned-k400",
    config=config
)

model = FightTimeSformer(backbone)
model.load_state_dict(
    torch.load("/content/drive/MyDrive/securevision_fight_timesformer_hardneg.pth", map_location=device)
)

model = model.to(device)
model.eval()
model.backbone.eval()

print("Model loaded successfully")


Model loaded successfully


In [ ]:
transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


In [ ]:
def sample_frames(vr, num_frames=8):
    total_frames = len(vr)

    if total_frames <= num_frames:
        indices = np.linspace(0, total_frames - 1, num_frames).astype(int)
    else:
        start = (total_frames - num_frames) // 2
        indices = np.arange(start, start + num_frames)

    frames = vr.get_batch(indices).asnumpy()
    frames = [transform(f) for f in frames]

    video = torch.stack(frames)           # [T, C, H, W]
    video = video.permute(1, 0, 2, 3)     # [C, T, H, W]

    return video


In [ ]:
import torch.nn.functional as F

def predict_video(video_path, threshold=0.5):
    vr = VideoReader(video_path, ctx=cpu(0))

    video = sample_frames(vr)
    video = video.unsqueeze(0).to(device)   # [1, C, T, H, W]
    video = video.permute(0, 2, 1, 3, 4)     # [1, T, C, H, W]

    with torch.no_grad():
        logits = model(video)
        probs = F.softmax(logits, dim=1)
        conf, pred = torch.max(probs, dim=1)

    label_map = {0: "NonViolence", 1: "Violence"}

    return {
        "prediction": label_map[pred.item()],
        "confidence": conf.item()
    }


def predict_video_multiclip(video_path, clips=5):
    vr = VideoReader(video_path, ctx=cpu(0))
    total_frames = len(vr)

    probs_all = []

    for i in range(clips):
        if total_frames > 8:
            start = int(i * (total_frames - 8) / clips)
            indices = np.arange(start, start + 8)
        else:
            indices = np.linspace(0, total_frames - 1, 8).astype(int)

        frames = vr.get_batch(indices).asnumpy()
        frames = [transform(f) for f in frames]
        video = torch.stack(frames).permute(1, 0, 2, 3)
        video = video.unsqueeze(0).to(device)
        video = video.permute(0, 2, 1, 3, 4)

        with torch.no_grad():
            logits = model(video)
            probs = torch.softmax(logits, dim=1)
            probs_all.append(probs.cpu())

    avg_probs = torch.mean(torch.cat(probs_all), dim=0)
    conf, pred = torch.max(avg_probs, dim=0)

    label_map = {0: "NonViolence", 1: "Violence"}

    return {
        "prediction": label_map[pred.item()],
        "confidence": conf.item()
    }



In [ ]:
!pip install -q decord

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 60.7 MB/s eta 0:00:00


In [ ]:
from decord import VideoReader, cpu

In [ ]:
video_path = "/content/drive/MyDrive/NV_34.mp4"  # change this

result = predict_video_multiclip(video_path)

print("Prediction:", result["prediction"])
print("Confidence:", round(result["confidence"] * 100, 2), "%")


Prediction: NonViolence
Confidence: 99.87 %
